# Figure: manual field vegetation measurements in winter wheat

Leaf area index, canopy height, aboveground dry biomass, crop N uptake, N
content and &#948;&#185;&#8309;N from the destructive samplings per treatment.

**Input** `../data/raw/` (LAI, canopy height, biomass)
**Output** `../figures/veg_measurements_wheat.png`,
`../figures/N_diagnostics_wheat.png`


# LOAD LIBRARIES

In [ ]:
import numpy as np
import pandas as pd # pandas allows to do a lot of basic data manipulation such as loading csv data files
from datetime import datetime
import matplotlib.pyplot as plt # matplotlib is a basic ploting library of Python
import matplotlib.dates as mdates
from scipy.stats import t

# LOAD DATA

In [ ]:
# create a dictionary of the file paths
directory_path = '../data/raw/'
files_dict = {
    'lai' : "LAI",
    'canopy_height' : "canopy_height_agg",
    'crop_biomass' : "crop_biomass_agg",
}

dfs = {}
for key, fname in files_dict.items():
    path = directory_path + fname + '.csv'
    df = pd.read_csv(path)
    df["source"] = key
    dfs[key] = df

df = pd.concat(dfs.values(), ignore_index=True, sort=False)
df['date'] = pd.to_datetime(df['date'])
df.set_index('date', inplace=True)

# Convert unit for biomass from kg/ha to t/ha
df['dbm_mean'] = df['dbm_mean'] / 1000
df['dbm_std'] = df['dbm_std'] / 1000
df['dbm_se'] = df['dbm_se'] / 1000

df = df.sort_index()  # sort by date

display(df.columns.tolist())
display(df)

# PLOTS

## CONFIGS

In [ ]:
# Period to plot
start_date = '2024-01-01'
end_date = '2024-07-25'
start_ts = pd.to_datetime(start_date)
end_ts = pd.to_datetime(end_date)

# Labels for y axis
y_labels = {
    'crop_n': 'Crop N (kg N ha$^{-1}$)',
    'dbm': 'Crop biomass (t DM ha$^{-1}$)',
    'LAI': 'Leaf Area Index (m$^2$ m$^{-2}$)',
    'can_height': 'Canopy height (cm)',
    'n_percent': 'Crop N concentration (%)',
    'd15n': '$\delta^{15}$N (‰)',
    'c_n_ratio': 'C/N ratio'
}

# Plotting configurations from plot_config.py
from plot_config import COLORS, LABELS, EVENT_LINES, add_event_lines

# Add the anthesis (1 June) to the event lines
anthesis = {
    'date': '2024-06-01',
    'color': 'gold',
    'linestyle': 'dashdot',
    'label_parts': [{'text': 'anthesis', 'color': 'black'}]
}
EVENT_LINES.append(anthesis)


In [ ]:
def plot_manual_measurements(df, start_date, end_date, cols):
    ncols = 2 if len(cols) > 3 else 1
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 4*nrows), sharex=True)
    plt.subplots_adjust(hspace=0.05, wspace=0.15) # reduce spacing between panels
    axes = np.atleast_2d(axes)  
    ax_list = axes.ravel(order='F')

    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)

    for k, col in enumerate(cols):
        ax = ax_list[k]

        # right column -> put y-axis on the right
        if ncols == 2 and k >= nrows:
            ax.yaxis.tick_right()
            ax.yaxis.set_label_position("right")
            ax.tick_params(axis="y", labelleft=False, labelright=True)
        else:                  # left column
            ax.yaxis.tick_left()
            ax.yaxis.set_label_position("left")
            ax.tick_params(axis="y", labelleft=True, labelright=False)

        mcol = f"{col}_mean"
        ecol = f"{col}_se"
        ncol = f"{col}_count"

        for parcel in ("A", "B"):
            # include ncol if present
            use_cols = [mcol, ecol] + ([ncol] if ncol in df.columns else [])
            sub = df.loc[df["parcel"] == parcel, use_cols].copy()
            sub = sub.loc[start_date:end_date].dropna(subset=[mcol, ecol])
            color = COLORS[parcel]
            mean = sub[mcol]
            se   = sub[ecol].fillna(0)
            err = se
            ax.plot(mean.index, mean, color=color, marker="o",
                    label=LABELS[parcel])
            ax.errorbar(
                mean.index, mean,
                yerr=err,
                fmt="none",
                ecolor=color,
                elinewidth=1.0,
                capsize=4
            )
            ax.fill_between(mean.index, mean - err, mean + err,
                            color=color, alpha=0.2, linewidth=0)

        ax.set_ylabel(y_labels[col] if col in y_labels else col)

        # Management events (lines on every panel)
        add_event_lines(ax, start_ts=start_ts, end_ts=end_ts, mgmt=False, height=1.02, x_spacing_days=6)

    # Event labels on top left panel
    ax_top = ax_list[0]  # top-left panel
    add_event_lines(ax_top, start_ts=start_ts, end_ts=end_ts, mgmt=True, height=1.02, x_spacing_days=6)

    # Legend only once
    if ncols > 1:  # if 2 cols, move legend in the second column to avoid overlap with management event labels
        x = 1
        y = 1.3
        ax_list[2].legend(frameon=False, loc="upper right", bbox_to_anchor=(x, y))
    else:
        ax_list[0].legend(frameon=False, loc="upper right")

    # Neat date axis on bottom row panels that are on
    for ax in axes[-1, :]:
        if ax.axison:
            ax.xaxis.set_major_locator(mdates.MonthLocator())
            ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))
            ax.tick_params(axis="x", labelrotation=0)

    # Align y-labels
    fig.align_ylabels(axes[:, 0])
    if ncols == 2:
        fig.align_ylabels(axes[:, 1])

    # Add letters at the top left of each panel
    for i, ax in enumerate(['(a)', '(b)', '(c)', '(d)']):
        if i < len(ax_list):
            ax_list[i].text(0.01, 0.99, ax, transform=ax_list[i].transAxes, fontweight='bold', verticalalignment='top', horizontalalignment='left')
    
    return fig, axes, ax_list

In [ ]:
# Vegetation measurements
incols = ['LAI', 'can_height', 'dbm', 'crop_n']
plt.rcParams.update({'font.size': 12})
fig, axes, ax_list = plot_manual_measurements(df, start_date, end_date, incols)
# Modify y-axis ticks for the biomass panel
top_right_ax = axes[0, 1] 
top_right_ax.set_yticks(range(0, 21, 5))
save_path ='../figures/veg_measurements_wheat.png'
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Vegetation measurements
incols = ['n_percent', 'd15n']
plt.rcParams.update({'font.size': 12})
fig, axes, ax_list = plot_manual_measurements(df, start_date, end_date, incols)
x, y = 1, 1.5
ax_list[0].legend(frameon=False, loc="upper right", bbox_to_anchor=(x, y)) 

fig.set_size_inches(12, 8, forward=True)
save_path='../figures/N_diagnostics_wheat.png'
plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()